In [ ]:
import scipy.stats as stats

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import re

In [ ]:
import numpy as np
import math

In [ ]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd 
from datetime import datetime
from  datetime import date
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.metrics import ndcg_score

In [ ]:
import sys
import psutil 

In [ ]:
X_train_merged_df_men = pd.read_excel('X_train_merged_df_men.xlsx')
X_test_merged_df_men = pd.read_excel('X_test_merged_df_men.xlsx')
X_train_merged_df_men=X_train_merged_df_men.drop(['Unnamed: 0'], axis=1)
X_test_merged_df_men=X_test_merged_df_men.drop(['Unnamed: 0'], axis=1)

In [ ]:
X_test_merged_df_men_copy = X_test_merged_df_men.copy() 

In [ ]:
y_train_merged_df_men = X_train_merged_df_men['Relevance']
y_test_merged_df_men = X_test_merged_df_men['Relevance']
X_train_merged_df_men = X_train_merged_df_men.drop('Relevance', axis=1)
X_test_merged_df_men = X_test_merged_df_men.drop('Relevance', axis=1)

In [ ]:
X_train_merged_df_women = pd.read_excel('X_train_merged_df_women.xlsx')
X_test_merged_df_women = pd.read_excel('X_test_merged_df_women.xlsx')
X_train_merged_df_women=X_train_merged_df_women.drop(['Unnamed: 0'], axis=1)
X_test_merged_df_women=X_test_merged_df_women.drop(['Unnamed: 0'], axis=1)

In [ ]:
X_test_merged_df_women_copy = X_test_merged_df_women.copy() 

In [ ]:
y_train_merged_df_women = X_train_merged_df_women['Relevance']
y_test_merged_df_women = X_test_merged_df_women['Relevance']
X_train_merged_df_women = X_train_merged_df_women.drop('Relevance', axis=1)
X_test_merged_df_women = X_test_merged_df_women.drop('Relevance', axis=1)

# NEW: hyperparameter tuning for Random Forest Regression, matching the
# validation-NDCG@10 selection criterion already used for LambdaRank, ListNet and ListMLE.

In [ ]:
def ndcg10_scorer(estimator, X, y):
    preds = estimator.predict(X)
    df = X.copy()
    df["_true"] = y.values if hasattr(y, "values") else y
    df["_pred"] = preds
    scores = []
    for _, group in df.groupby("Event_ID"):
        if len(group) < 2:
            continue
        y_true = group["_true"].to_numpy().reshape(1, -1)
        y_score = group["_pred"].to_numpy().reshape(1, -1)
        scores.append(ndcg_score(y_true, y_score, k=10))
    return np.mean(scores)

param_grid_rf = {
    "n_estimators": np.arange(100, 700, 50),
    "max_depth": [4, 6, 8, 10, 12, 16, None],
    "min_samples_leaf": np.arange(1, 12),
    "min_samples_split": np.arange(2, 20, 2),
    "max_features": ["sqrt", "log2", 0.5, 0.8, 1.0],
}


# GroupKFold keeps every athlete from the same
# event in the same fold, so no event is split across train/validation.

In [ ]:
rf_search_men = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_grid_rf,
    n_iter=25,
    scoring=ndcg10_scorer,
    cv=GroupKFold(n_splits=4).split(
        X_train_merged_df_men, y_train_merged_df_men,
        groups=X_train_merged_df_men["Event_ID"],
    ),
    random_state=42,
    n_jobs=-1,
)
rf_search_men.fit(X_train_merged_df_men, y_train_merged_df_men)
best_params_rf_men = rf_search_men.best_params_
print("Best params (RF, men):", best_params_rf_men)
print("Best CV NDCG@10 (RF, men):", rf_search_men.best_score_)

In [ ]:
rf_search_women = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_grid_rf,
    n_iter=25,
    scoring=ndcg10_scorer,
    cv=GroupKFold(n_splits=4).split(
        X_train_merged_df_women, y_train_merged_df_women,
        groups=X_train_merged_df_women["Event_ID"],
    ),
    random_state=42,
    n_jobs=-1,
)
rf_search_women.fit(X_train_merged_df_women, y_train_merged_df_women)
best_params_rf_women = rf_search_women.best_params_
print("Best params (RF, women):", best_params_rf_women)
print("Best CV NDCG@10 (RF, women):", rf_search_women.best_score_)

In [ ]:
model_tuned_men = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params_rf_men)
model_tuned_men.fit(X_train_merged_df_men, y_train_merged_df_men)
y_pred_tuned_men = model_tuned_men.predict(X_test_merged_df_men)

In [ ]:
model_tuned_women = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params_rf_women)
model_tuned_women.fit(X_train_merged_df_women, y_train_merged_df_women)
y_pred_tuned_women = model_tuned_women.predict(X_test_merged_df_women)

In [ ]:
# Add true and predicted values
X_test_merged_df_men_copy["Relevance"] = y_test_merged_df_men.values
X_test_merged_df_men_copy["Predicted_tuned"] = y_pred_tuned_men

constant_true_events = []
constant_pred_events = []

for event_id, group in X_test_merged_df_men_copy.groupby("Event_ID"):
    y_true = group["Relevance"].to_numpy()
    y_pred_event = group["Predicted_tuned"].to_numpy()

    if np.unique(y_true).size == 1:
        constant_true_events.append(event_id)

    if np.unique(y_pred_event).size == 1:
        constant_pred_events.append(event_id)

print(f"Events with constant true relevance: {len(constant_true_events)}")
print(f"Events with constant predictions: {len(constant_pred_events)}")

if constant_true_events:
    print("Constant true events:", constant_true_events[:10])

if constant_pred_events:
    print("Constant prediction events:", constant_pred_events[:10])

In [ ]:
# Add true and predicted values
X_test_merged_df_women_copy["Relevance"] = y_test_merged_df_women.values
X_test_merged_df_women_copy["Predicted_tuned"] = y_pred_tuned_women

constant_true_events = []
constant_pred_events = []

for event_id, group in X_test_merged_df_women_copy.groupby("Event_ID"):
    y_true = group["Relevance"].to_numpy()
    y_pred_event = group["Predicted_tuned"].to_numpy()

    if np.unique(y_true).size == 1:
        constant_true_events.append(event_id)

    if np.unique(y_pred_event).size == 1:
        constant_pred_events.append(event_id)

print(f"Events with constant true relevance: {len(constant_true_events)}")
print(f"Events with constant predictions: {len(constant_pred_events)}")

if constant_true_events:
    print("Constant true events:", constant_true_events[:10])

if constant_pred_events:
    print("Constant prediction events:", constant_pred_events[:10])

In [ ]:
scores = []

for event_id, group in X_test_merged_df_men_copy.groupby("Event_ID"):
    y_true = group["Relevance"].to_numpy()
    y_pred_event = group["Predicted_tuned"].to_numpy()

    if np.unique(y_true).size < 2 or np.unique(y_pred_event).size < 2:
        continue
    rho, _ = spearmanr(y_true, y_pred_event)
    scores.append(rho)

mean_spearman = np.mean(scores)
std_spearman = np.std(scores)

In [ ]:
print(f"Mean spearman: {mean_spearman:.4f}")
print(f"Std spearman: {std_spearman:.4f}")

In [ ]:
scores = []

for event_id, group in X_test_merged_df_women_copy.groupby("Event_ID"):
    y_true = group["Relevance"].to_numpy()
    y_pred_event = group["Predicted_tuned"].to_numpy()

    if np.unique(y_true).size < 2 or np.unique(y_pred_event).size < 2:
        continue
    rho, _ = spearmanr(y_true, y_pred_event)
    scores.append(rho)

mean_spearman = np.mean(scores)
std_spearman = np.std(scores)

In [ ]:
print(f"Mean spearman: {mean_spearman:.4f}")
print(f"Std spearman: {std_spearman:.4f}")

In [ ]:
ndcg1 = []
ndcg2 = []
ndcg3 = []
ndcg4 = []

for _, group in X_test_merged_df_men_copy.groupby("Event_ID"):

    y_true = group["Relevance"].values
    y_score = group["Predicted_tuned"].values
    n = len(group)

    ndcg1.append(ndcg_score([y_true], [y_score], k=1))
    if n >= 2:
        ndcg2.append(ndcg_score([y_true], [y_score], k=2))
    if n >= 3:
        ndcg3.append(ndcg_score([y_true], [y_score], k=3))
    if n >= 4:
        ndcg4.append(ndcg_score([y_true], [y_score], k=4))

print(f"Mean NDCG@1: {np.mean(ndcg1):.4f}")
print(f"Mean NDCG@2: {np.mean(ndcg2):.4f}")
print(f"Mean NDCG@3: {np.mean(ndcg3):.4f}")
print(f"Mean NDCG@4: {np.mean(ndcg4):.4f}")

In [ ]:
ndcg1 = []
ndcg2 = []
ndcg3 = []
ndcg4 = []

for _, group in X_test_merged_df_women_copy.groupby("Event_ID"):

    y_true = group["Relevance"].values
    y_score = group["Predicted_tuned"].values
    n = len(group)

    ndcg1.append(ndcg_score([y_true], [y_score], k=1))
    if n >= 2:
        ndcg2.append(ndcg_score([y_true], [y_score], k=2))
    if n >= 3:
        ndcg3.append(ndcg_score([y_true], [y_score], k=3))
    if n >= 4:
        ndcg4.append(ndcg_score([y_true], [y_score], k=4))

print(f"Mean NDCG@1: {np.mean(ndcg1):.4f}")
print(f"Mean NDCG@2: {np.mean(ndcg2):.4f}")
print(f"Mean NDCG@3: {np.mean(ndcg3):.4f}")
print(f"Mean NDCG@4: {np.mean(ndcg4):.4f}")

In [ ]:
X_test_merged_df_men_copy.to_excel("X_test_merged_df_men_copy_Random_FR_results.xlsx")
X_test_merged_df_women_copy.to_excel("X_test_merged_df_women_copy_Random_FR_results.xlsx")